# 25 — Bài báo, vòng sửa: đối chứng công bằng, chính sách định tuyến, gần trùng và chi phí (máy local)

**Phạm vi**

Phân tích khám phá theo amendment v4.2, chỉ dùng dự đoán đã lưu: không train lại, không gọi API, không đổi cấu hình hay prompt đã khóa.

**Nội dung**

B1 so mô hình phẳng argmax, mô hình phẳng có ngưỡng và cascade dưới cùng luật ngưỡng ưu tiên recall. B2 so định tuyến theo độ tin cậy, định tuyến
khi PhoBERT và ViSoBERT bất đồng, và định tuyến ngẫu nhiên cùng số lượt gọi LLM. A3 kiểm tra gần trùng với Train ∪ Validation. A6 bổ sung bảng tham số
và các kịch bản chi phí.

**Thứ tự**

Phần A chỉ đọc Validation, chọn ngưỡng B1 và chốt mọi hằng số, lưu kèm SHA-256. Phần B kiểm tra mã băm rồi mới đọc nhãn Hold-out, focused và Lazada;
đây là lần dùng Hold-out và Lazada thứ tư. Phần C chỉ dùng số liệu chi phí đã lưu, không đọc nhãn.

In [1]:
import os, sys, json, datetime, subprocess, hashlib
import numpy as np, pandas as pd, yaml
from scipy.stats import spearmanr
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.final_eval import load_frozen
from src.classification.evaluate import FOUR_CLASSES, OTHER_ID, metrics
from src.classification import paper_analysis as PA
from src.classification import routing_calibration as RC
from src.classification import fair_controls as FC
from src.classification import cost_model as CM
from src.classification.strategy_llm_fewshot.classify import cost_summary
from src.utils.llm import prices
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 300)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
FZ = load_frozen(ROOT)
SEEDS = FZ['tier1']['seeds']

def frozen_yaml(fname, key, must):
    txt = open(os.path.join(ROOT, '03_configs', 'frozen', fname), encoding='utf-8').read()
    assert txt.startswith('# FROZEN'), f'{fname}: thiếu header FROZEN'
    for m in must:
        assert m in txt, f'{fname}: thiếu dấu "{m}"'
    return yaml.safe_load(txt)[key]

TF = frozen_yaml('paper_tfidf_config.yaml', 'tfidf', ['amendment v3.2'])
NC = frozen_yaml('paper_neural_config.yaml', 'paper_neural', ['amendment v3.4', 'amendment v3.5'])
CFG_COST = frozen_yaml('paper_cost_model.yaml', 'cost_model', ['amendment v3.8'])
assert '| v4.2 |' in open(os.path.join(ROOT, '00_protocol', 'protocol_amendments.md'), encoding='utf-8').read()
STEP4 = json.load(open(os.path.join(RES, 'paper_step4_selection.json'), encoding='utf-8'))
print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT, '| seeds:', SEEDS)
print('θ của v4.1 (PhoBERT):', STEP4['routing']['PhoBERT']['theta_max'], STEP4['routing']['PhoBERT']['theta_98'])

Ngày chạy: 21/11/2026 | commit: b436fad136189c47383eb98f13dfdf1539e9c940 | seeds: [42, 1337, 2024]
θ của v4.1 (PhoBERT): 0.92 0.91


## Phần A — chọn trên Validation

**B1**

Luật tầng 1 hiện hành (v2.0 + chặn điểm biên v3.5) áp lên P(ACT) = 1 − P(OTHER) của từng mô hình phẳng, seed 42.

**B2**

Router độ tin cậy dùng θ của v4.1, không chọn lại. Router bất đồng không có tham số; tỷ lệ bất đồng trên Validation chỉ để mô tả.

In [2]:
VAL = pd.read_csv(os.path.join(DATA, 'shopee_validation.csv'), encoding='utf-8-sig')
VIDS = VAL['review_id'].tolist()
VY4 = VAL['gold_label'].map(RC.LBL).to_numpy()
FLAT_VAL = {x: FC.flat_probs(RES, 'validation', VIDS, m, SEEDS) for x, m in FC.TRAINED.items()}
B1_SEL, rows = {}, []
for x, runs in FLAT_VAL.items():
    c = FC.choose_flat_threshold(VY4, runs[42])
    B1_SEL[x] = c
    rows.append({'chiến lược': x, 't': c['threshold'], 'đạt sàn recall': c['recall_constraint_met'],
                 'recall ACT': c['recall_actionable'], 'precision ACT': c['precision_actionable'],
                 'Macro-F1 4 lớp argmax': metrics(VY4, runs[42].argmax(1), FOUR_CLASSES)['macro_f1'],
                 'Macro-F1 4 lớp có ngưỡng': metrics(VY4, FC.flat_threshold_predict(runs[42], c['threshold']), FOUR_CLASSES)['macro_f1']})
B1_VAL = pd.DataFrame(rows)
print('B1 trên Validation (seed 42):'); print(B1_VAL.round(3).to_string(index=False))

LLM_VAL = RC.flat_llm_validation(os.path.join(RES, 'paper_flat_llm_val_calls.jsonl'), VIDS)
dis = FC.disagreement_mask(FLAT_VAL['PhoBERT'][42], FLAT_VAL['ViSoBERT'][42])
f_dis = metrics(VY4, FC.route_mask(FLAT_VAL['PhoBERT'][42].argmax(1), LLM_VAL, dis), FOUR_CLASSES)['macro_f1']
print(f'\nB2 trên Validation (mô tả): bất đồng PhoBERT–ViSoBERT {dis.mean() * 100:.1f}% review, Macro-F1 router bất đồng {f_dis * 100:.1f}')

B1 trên Validation (seed 42):
        chiến lược    t  đạt sàn recall  recall ACT  precision ACT  Macro-F1 4 lớp argmax  Macro-F1 4 lớp có ngưỡng
            TF-IDF 0.49            True       0.831          0.578                  0.665                     0.652
Bi-LSTM + fastText 0.18            True       0.817          0.637                  0.685                     0.664
          ViSoBERT 0.06            True       0.817          0.659                  0.686                     0.694
           PhoBERT 0.08            True       0.803          0.750                  0.666                     0.654

B2 trên Validation (mô tả): bất đồng PhoBERT–ViSoBERT 8.3% review, Macro-F1 router bất đồng 73.2


In [3]:
SELECTION = {
    'amendment': 'v4.2',
    'b1': {x: {'threshold': c['threshold'], 'recall_constraint_met': bool(c['recall_constraint_met'])} for x, c in B1_SEL.items()},
    'b1_rule': 'guarded rule v2.0 + v3.5 on P(ACT) = 1 - P(OTHER), Validation, seed 42',
    'routing_v41': {'theta_max': STEP4['routing']['PhoBERT']['theta_max'], 'theta_98': STEP4['routing']['PhoBERT']['theta_98']},
    'budgets': list(FC.BUDGETS), 'random': {'draws': FC.RANDOM_DRAWS, 'seed': FC.RANDOM_SEED},
    'near_duplicates': {'method': 'char_wb 3-5-gram TF-IDF cosine on normalised text; reference = Train + Validation',
                        'main': FC.NEAR_MAIN, 'sensitivity': FC.NEAR_SENS},
    'cost': {'llm_price_multipliers': [1, 5, 20], 'quality_targets': [78, 79, 80], 'observed_reviews_per_year': 57200}}
SEL_PATH = os.path.join(RES, 'paper_step5_selection.json')
SEL_HASH = RC.save_selection(SELECTION, SEL_PATH)
print('Đã lưu lựa chọn trên Validation:', os.path.basename(SEL_PATH), '| SHA-256:', SEL_HASH)

Đã lưu lựa chọn trên Validation: paper_step5_selection.json | SHA-256: 4c19fb8f2227c73b49ad08ded6b7da8de6c7571de8df36483f2664b4865f2bb8


## Phần B — chấm trên các tập khóa

**Ràng buộc**

Lựa chọn của phần A được đọc lại và kiểm tra mã băm trước khi nạp nhãn. Xác suất phẳng đã lưu phải cho đúng nhãn argmax đã báo cáo ở bước 2.

In [4]:
assert hashlib.sha256(open(SEL_PATH, encoding='utf-8').read().encode('utf-8')).hexdigest() == SEL_HASH
SEL = json.load(open(SEL_PATH, encoding='utf-8'))
NAME = {'holdout': 'shopee_holdout.csv', 'lazada': 'lazada_external_test.csv', 'focused': 'shopee_focused_feature_ux.csv'}
D = {s: pd.read_csv(os.path.join(DATA, f), encoding='utf-8-sig') for s, f in NAME.items()}
IDS = {s: d['review_id'].tolist() for s, d in D.items()}
Y4 = {s: d['gold_label'].map(RC.LBL).to_numpy() for s, d in D.items()}
PRED = {s: PA.step2_predictions(RES, s, IDS[s], FZ, TF, NC) for s in NAME}
CFG = {s: PA.configurations_step2(*PRED[s], hybrids=False)[0] for s in NAME}
FLAT = {s: {x: FC.flat_probs(RES, s, IDS[s], m, SEEDS) for x, m in FC.TRAINED.items()} for s in NAME}
for s in NAME:
    for x in FC.TRAINED:
        assert all((FLAT[s][x][k].argmax(1) == PRED[s][2][x][k]).all() for k in SEEDS), (s, x)
print({s: len(IDS[s]) for s in NAME}, '| xác suất phẳng khớp dự đoán argmax đã báo cáo')

def act42(y, pred):
    g, a = np.asarray(y) != OTHER_ID, np.asarray(pred) != OTHER_ID
    tp = (a & g).sum()
    return tp / max(g.sum(), 1), tp / max(a.sum(), 1), a.mean()

def compare(y, diff, canon, pairs, family):
    rows = []
    for a, b in pairs:
        d, sg = PA._pair(diff, a, b)
        lo, hi = sorted([sg * d['CI 2,5%'], sg * d['CI 97,5%']])
        n01, n10, p = PA.mcnemar_exact(y, canon[a], canon[b])
        rows.append({'họ so sánh': family, 'A': a, 'B': b, 'chênh A − B (seed 42)': sg * d['chênh (A − B)'] * 100,
                     'CI 2,5%': lo * 100, 'CI 97,5%': hi * 100, 'A đúng, B sai': n01, 'A sai, B đúng': n10, 'p (exact)': p})
    df = pd.DataFrame(rows)
    df['p (Holm)'] = PA.holm(df['p (exact)'])
    return df

{'holdout': 1000, 'lazada': 500, 'focused': 150} | xác suất phẳng khớp dự đoán argmax đã báo cáo


## B1 — phẳng argmax, phẳng có ngưỡng và cascade

**Cách đọc**

Mọi điểm ước lượng chính lấy từ lần chạy seed 42; trung bình ± độ lệch chuẩn qua ba seed để trong cột riêng. "Chuyển xử lý" là tỷ lệ review được dự
đoán actionable. Focused set chỉ mô tả.

In [5]:
def b1(s, test=True):
    cfg = {}
    for x in FC.TRAINED:
        t = SEL['b1'][x]['threshold']
        cfg[f'Phẳng {x}'] = CFG[s][f'Phẳng {x}']
        cfg[f'Phẳng {x} (ngưỡng)'] = {k: FC.flat_threshold_predict(p, t) for k, p in FLAT[s][x].items()}
        cfg[f'Cascade {x}'] = CFG[s][f'Cascade {x}']
    tab, diff, y, canon = PA.e2e_scores(Y4[s], cfg)
    a42 = {n: act42(y, canon[n]) for n in cfg}
    tab['Recall ACT (seed 42)'] = tab['cấu hình'].map(lambda n: a42[n][0])
    tab['Precision ACT (seed 42)'] = tab['cấu hình'].map(lambda n: a42[n][1])
    tab['chuyển xử lý (seed 42)'] = tab['cấu hình'].map(lambda n: a42[n][2])
    tab['chuyển xử lý (TB 3 seed)'] = tab['cấu hình'].map(lambda n: FC.forward_rate(cfg[n]))
    tab.insert(0, 'tập', s)
    cmp = None
    if test:
        pairs = [p for x in FC.TRAINED for p in ((f'Phẳng {x} (ngưỡng)', f'Phẳng {x}'), (f'Cascade {x}', f'Phẳng {x} (ngưỡng)'))]
        cmp = compare(y, diff, canon, pairs, f'B1 – {s}')
    return tab, cmp

B1_TAB, B1_CMP = {}, {}
for s in NAME:
    B1_TAB[s], B1_CMP[s] = b1(s, test=(s != 'focused'))
    v = B1_TAB[s][['cấu hình', 'Macro-F1 chuẩn', 'CI 2,5%', 'CI 97,5%', 'Macro-F1 (4 lớp)', 'F1 FEATURE', 'Recall ACT (seed 42)',
                   'Precision ACT (seed 42)', 'chuyển xử lý (seed 42)']].copy()
    for c in ['Macro-F1 chuẩn', 'CI 2,5%', 'CI 97,5%', 'Recall ACT (seed 42)', 'Precision ACT (seed 42)', 'chuyển xử lý (seed 42)']:
        v[c] = (v[c] * 100).round(1)
    print(f'\n=== B1 – {s} ==='); print(v.to_string(index=False))
    if B1_CMP[s] is not None:
        print(B1_CMP[s].round(4).to_string(index=False))


=== B1 – holdout ===
                         cấu hình  Macro-F1 chuẩn  CI 2,5%  CI 97,5% Macro-F1 (4 lớp) F1 FEATURE  Recall ACT (seed 42)  Precision ACT (seed 42)  chuyển xử lý (seed 42)
                     Phẳng TF-IDF            62.8     57.5      67.6       62.8 ± 0.0 28.6 ± 0.0                  65.6                     68.6                    17.5
            Phẳng TF-IDF (ngưỡng)            60.8     56.7      64.6       60.8 ± 0.0 17.6 ± 0.0                  82.5                     54.9                    27.5
                   Cascade TF-IDF            61.3     56.9      65.5       61.3 ± 0.0 21.3 ± 0.0                  78.7                     60.0                    24.0
         Phẳng Bi-LSTM + fastText            62.9     56.4      68.7       59.7 ± 4.0 25.8 ± 8.6                  61.7                     73.4                    15.4
Phẳng Bi-LSTM + fastText (ngưỡng)            61.7     55.9      67.1       52.0 ± 9.3 19.2 ± 9.6                  83.1                    


=== B1 – lazada ===
                         cấu hình  Macro-F1 chuẩn  CI 2,5%  CI 97,5% Macro-F1 (4 lớp) F1 FEATURE  Recall ACT (seed 42)  Precision ACT (seed 42)  chuyển xử lý (seed 42)
                     Phẳng TF-IDF            56.1     48.9      62.6       56.1 ± 0.0 19.6 ± 0.0                  70.4                     50.0                    22.8
            Phẳng TF-IDF (ngưỡng)            55.7     49.5      61.2       55.7 ± 0.0 29.3 ± 0.0                  88.9                     37.5                    38.4
                   Cascade TF-IDF            56.6     50.2      62.5       56.6 ± 0.0 26.7 ± 0.0                  88.9                     41.9                    34.4
         Phẳng Bi-LSTM + fastText            54.6     47.0      62.0       50.7 ± 4.1 20.7 ± 5.3                  69.1                     50.9                    22.0
Phẳng Bi-LSTM + fastText (ngưỡng)            50.1     43.2      56.8       42.9 ± 7.2 17.2 ± 4.8                  92.6                     


=== B1 – focused ===
                         cấu hình  Macro-F1 chuẩn  CI 2,5%  CI 97,5% Macro-F1 (4 lớp) F1 FEATURE  Recall ACT (seed 42)  Precision ACT (seed 42)  chuyển xử lý (seed 42)
                     Phẳng TF-IDF            61.4     51.7      70.2       61.4 ± 0.0 75.6 ± 0.0                  90.1                     77.4                    70.7
            Phẳng TF-IDF (ngưỡng)            57.8     47.8      66.6       57.8 ± 0.0 70.9 ± 0.0                  95.6                     70.7                    82.0
                   Cascade TF-IDF            53.2     44.4      61.6       53.2 ± 0.0 68.9 ± 0.0                  96.7                     70.4                    83.3
         Phẳng Bi-LSTM + fastText            61.2     52.5      69.2       62.0 ± 1.7 70.0 ± 5.1                  75.8                     88.5                    52.0
Phẳng Bi-LSTM + fastText (ngưỡng)            56.1     48.5      63.2       53.4 ± 2.8 70.1 ± 2.6                  92.3                    

## B2 — chính sách định tuyến sang LLM

**Router**

Độ tin cậy tại θ_max và θ_98 (v4.1); bất đồng giữa PhoBERT và ViSoBERT phẳng; độ tin cậy gửi đúng bằng số review bất đồng; độ tin cậy ở các ngân sách
cố định. Mỗi router được so với định tuyến ngẫu nhiên cùng số lượt gọi trên từng seed. Đây là đánh giá bằng đầu ra LLM lưu sẵn; chi phí API tính theo số
lượt chuyển của từng chính sách.

In [6]:
LLM_USD = {s: cost_summary(pd.read_csv(os.path.join(RES, f'paper_flat_llm_{s}.csv')), prices(), s)['usd_/1000_review'] / 1000
           for s in ('holdout', 'lazada')}
TH = SEL['routing_v41']
CONF_MAX, CONF_98, DIS, CONF_DIS = 'PhoBERT ⇢ LLM (θ_max)', 'PhoBERT ⇢ LLM (θ_98)', 'PhoBERT ⇢ LLM (bất đồng)', 'PhoBERT ⇢ LLM (độ tin cậy, ngân sách bất đồng)'

def b2(s):
    y, llm = Y4[s], PA.canonical(PRED[s][2][PA.LLM])
    pho, vis = FLAT[s]['PhoBERT'], FLAT[s]['ViSoBERT']
    n = len(y)
    masks = {CONF_MAX: {k: FC.confidence(p) < TH['theta_max'] for k, p in pho.items()},
             CONF_98: {k: FC.confidence(p) < TH['theta_98'] for k, p in pho.items()},
             DIS: {k: FC.disagreement_mask(pho[k], vis[k]) for k in SEEDS}}
    masks[CONF_DIS] = {k: FC.lowest_confidence_mask(pho[k], int(masks[DIS][k].sum())) for k in SEEDS}
    for b in SEL['budgets']:
        masks[f'PhoBERT ⇢ LLM (độ tin cậy, {int(round(b * 100))}%)'] = {k: FC.lowest_confidence_mask(pho[k], int(round(b * n))) for k in SEEDS}
    cfg = {'Phẳng PhoBERT': CFG[s]['Phẳng PhoBERT'], 'Phẳng LLM few-shot': CFG[s]['Phẳng LLM few-shot']}
    for name, m in masks.items():
        cfg[name] = {k: FC.route_mask(pho[k].argmax(1), llm, m[k]) for k in SEEDS}
    tab, diff, yy, canon = PA.e2e_scores(y, cfg)
    fixed_rate = {'Phẳng PhoBERT': 0.0, 'Phẳng LLM few-shot': 1.0}
    rate42 = {**fixed_rate, **{name: float(m[42].mean()) for name, m in masks.items()}}
    tab['gửi LLM (seed 42)'] = tab['cấu hình'].map(rate42)
    tab['gửi LLM (TB 3 seed)'] = tab['cấu hình'].map(lambda c: float(np.mean([m.mean() for m in masks[c].values()])) if c in masks else fixed_rate[c])
    tab['USD API / 1.000 review (seed 42)'] = tab['gửi LLM (seed 42)'] * LLM_USD[s] * 1000
    tab.insert(0, 'tập', s)
    rnd = []
    for name, m in masks.items():
        for k in SEEDS:
            pred = FC.route_mask(pho[k].argmax(1), llm, m[k])
            f = metrics(y, pred, FOUR_CLASSES)['macro_f1']
            rnd.append({'tập': s, 'router': name, 'seed': k, 'Macro-F1 router': f,
                        **FC.random_baseline(y, pho[k].argmax(1), llm, int(m[k].sum()), f)})
    pairs = [(CONF_DIS, DIS)] + [(r, ref) for r in (CONF_MAX, CONF_98, DIS) for ref in ('Phẳng PhoBERT', 'Phẳng LLM few-shot')]
    cmp = compare(yy, diff, canon, pairs, f'B2 – {s}')
    return tab, pd.DataFrame(rnd), cmp

B2_TAB, B2_RND, B2_CMP = {}, {}, {}
for s in ('holdout', 'lazada'):
    B2_TAB[s], B2_RND[s], B2_CMP[s] = b2(s)
    v = B2_TAB[s][['cấu hình', 'Macro-F1 chuẩn', 'CI 2,5%', 'CI 97,5%', 'Macro-F1 (4 lớp)', 'gửi LLM (seed 42)', 'gửi LLM (TB 3 seed)',
                   'USD API / 1.000 review (seed 42)']].copy()
    for c in ['Macro-F1 chuẩn', 'CI 2,5%', 'CI 97,5%', 'gửi LLM (seed 42)', 'gửi LLM (TB 3 seed)']:
        v[c] = (v[c] * 100).round(1)
    print(f'\n=== B2 – {s} ==='); print(v.round(3).to_string(index=False))
    r = B2_RND[s][B2_RND[s]['seed'] == 42].copy()
    for c in ['Macro-F1 router', 'random mean', 'random SD', 'random P2.5', 'random P97.5']:
        r[c] = (r[c] * 100).round(1)
    print('Router so với định tuyến ngẫu nhiên cùng số lượt gọi (seed 42):'); print(r.drop(columns=['tập', 'seed']).to_string(index=False))
    print(B2_CMP[s].round(4).to_string(index=False))


=== B2 – holdout ===
                                      cấu hình  Macro-F1 chuẩn  CI 2,5%  CI 97,5% Macro-F1 (4 lớp)  gửi LLM (seed 42)  gửi LLM (TB 3 seed)  USD API / 1.000 review (seed 42)
                                 Phẳng PhoBERT            70.8     64.6      76.0       70.9 ± 1.0                0.0                  0.0                             0.000
                            Phẳng LLM few-shot            80.0     75.1      84.2             80.0              100.0                100.0                             0.266
                         PhoBERT ⇢ LLM (θ_max)            79.9     74.8      84.2       80.1 ± 0.3               12.2                 17.4                             0.032
                          PhoBERT ⇢ LLM (θ_98)            79.1     73.8      83.6       79.7 ± 0.6               10.6                 14.8                             0.028
                      PhoBERT ⇢ LLM (bất đồng)            77.8     72.1      82.4       78.2 ± 0.7               


=== B2 – lazada ===
                                      cấu hình  Macro-F1 chuẩn  CI 2,5%  CI 97,5% Macro-F1 (4 lớp)  gửi LLM (seed 42)  gửi LLM (TB 3 seed)  USD API / 1.000 review (seed 42)
                                 Phẳng PhoBERT            67.2     58.4      74.5       67.4 ± 2.1                0.0                  0.0                             0.000
                            Phẳng LLM few-shot            59.5     51.3      66.6             59.5              100.0                100.0                             0.268
                         PhoBERT ⇢ LLM (θ_max)            67.0     58.1      74.4       64.1 ± 2.6               19.2                 28.1                             0.051
                          PhoBERT ⇢ LLM (θ_98)            69.8     61.1      77.0       66.6 ± 2.9               16.8                 23.4                             0.045
                      PhoBERT ⇢ LLM (bất đồng)            72.8     64.6      79.5       69.0 ± 3.8                

## A3 — gần trùng với Train ∪ Validation

**Quy tắc**

Cosine TF-IDF ký tự 3–5-gram trên văn bản chuẩn hóa; gần trùng chính ≥ 0,90 (không tính trùng y hệt), độ nhạy ≥ 0,80. Không in văn bản review.

In [7]:
TRAIN = pd.read_csv(os.path.join(DATA, 'shopee_train.csv'), encoding='utf-8-sig')
REF = pd.concat([TRAIN['text'], VAL['text']]).astype(str)
ND, rows = {}, []
for s in NAME:
    nd = PA.near_duplicates(REF, D[s]['text'].astype(str), FC.NEAR_MAIN)
    nd['near_080'] = (nd['max_cosine'] >= FC.NEAR_SENS) & ~nd['exact']
    nd['gold'] = D[s]['gold_label'].to_numpy()
    nd['n_tokens'] = D[s]['text'].astype(str).str.split().str.len().to_numpy()
    ND[s] = nd
    for tag, m in (('trùng y hệt', nd['exact']), ('gần trùng ≥ 0,90', nd['near']), ('gần trùng ≥ 0,80', nd['near_080'])):
        rows.append({'tập': s, 'loại': tag, 'số review': int(m.sum()), 'tỷ lệ (%)': round(m.mean() * 100, 1),
                     'actionable': int((m & (nd['gold'] != 'OTHER')).sum()),
                     'số từ trung vị': float(nd.loc[m, 'n_tokens'].median()) if m.any() else np.nan})
NEAR_SUM = pd.DataFrame(rows)
print(NEAR_SUM.to_string(index=False))

h = 'holdout'
keep = ~(ND[h]['exact'] | ND[h]['near']).to_numpy()
pure = {k: v for k, v in CFG[h].items()}
pure[CONF_MAX] = {k: FC.route_mask(FLAT[h]['PhoBERT'][k].argmax(1), PA.canonical(PRED[h][2][PA.LLM]), FC.confidence(FLAT[h]['PhoBERT'][k]) < TH['theta_max']) for k in SEEDS}
pure[CONF_98] = {k: FC.route_mask(FLAT[h]['PhoBERT'][k].argmax(1), PA.canonical(PRED[h][2][PA.LLM]), FC.confidence(FLAT[h]['PhoBERT'][k]) < TH['theta_98']) for k in SEEDS}
full = PA.e2e_scores(Y4[h], pure)[0].set_index('cấu hình')
dedup = PA.e2e_scores(Y4[h], pure, mask=keep)[0].set_index('cấu hình')
NEAR_E2E = pd.DataFrame({'Macro-F1 đầy đủ (seed 42)': full['Macro-F1 chuẩn'] * 100, 'Macro-F1 bỏ trùng và gần trùng (seed 42)': dedup['Macro-F1 chuẩn'] * 100})
NEAR_E2E['chênh'] = NEAR_E2E.iloc[:, 1] - NEAR_E2E.iloc[:, 0]
NEAR_E2E = NEAR_E2E.reset_index()
rho = spearmanr(NEAR_E2E.iloc[:, 1], NEAR_E2E.iloc[:, 2]).correlation
print(f'\nHold-out còn {keep.sum()} review sau khi bỏ trùng y hệt và gần trùng ≥ 0,90; Spearman thứ hạng {rho:.3f}')
print(NEAR_E2E.round(1).to_string(index=False))

    tập             loại  số review  tỷ lệ (%)  actionable  số từ trung vị
holdout      trùng y hệt        217       21.7           2             1.0
holdout gần trùng ≥ 0,90         17        1.7           3             3.0
holdout gần trùng ≥ 0,80         48        4.8           7             3.0
 lazada      trùng y hệt         79       15.8           1             1.0
 lazada gần trùng ≥ 0,90          3        0.6           1             3.0
 lazada gần trùng ≥ 0,80         11        2.2           3             3.0
focused      trùng y hệt          1        0.7           0             2.0
focused gần trùng ≥ 0,90          2        1.3           1             3.0
focused gần trùng ≥ 0,80          5        3.3           2             3.0



Hold-out còn 766 review sau khi bỏ trùng y hệt và gần trùng ≥ 0,90; Spearman thứ hạng 0.979
                  cấu hình  Macro-F1 đầy đủ (seed 42)  Macro-F1 bỏ trùng và gần trùng (seed 42)  chênh
            Cascade TF-IDF                       61.3                                      60.1   -1.3
Cascade Bi-LSTM + fastText                       54.2                                      52.6   -1.5
          Cascade ViSoBERT                       62.9                                      61.6   -1.3
           Cascade PhoBERT                       61.7                                      60.3   -1.4
      Cascade LLM few-shot                       81.3                                      80.7   -0.6
              Phẳng TF-IDF                       62.8                                      62.0   -0.7
  Phẳng Bi-LSTM + fastText                       62.9                                      61.8   -1.1
            Phẳng ViSoBERT                       66.7                              

## Phần C — chi phí (không đọc nhãn)

**Nội dung**

Bảng tham số; kịch bản tổ chức đã có nhãn; giá LLM nhân 1, 5 và 20; định tuyến theo tỷ lệ gọi thực tế từng tập; quy mô quan sát 57.200 review/năm;
độ nhạy ngưỡng chất lượng và lựa chọn giữa Macro-F1 seed 42 với trung bình ba seed.

In [8]:
B = CFG_COST['base_scenario']
COMP = pd.read_csv(os.path.join(RES, 'paper_cost_components.csv'), encoding='utf-8-sig')
RCOMP = pd.read_csv(os.path.join(RES, 'paper_step4_routing_cost_components.csv'), encoding='utf-8-sig')
COMP2 = pd.concat([COMP, RCOMP], ignore_index=True)
for c in ('gọi API', 'có huấn luyện'):
    COMP2[c] = COMP2[c].astype(str).str.lower().eq('true')
d = CFG_COST['drift']
PARAMS = pd.DataFrame([
    ('c_L', 'chi phí một nhãn', 'USD/nhãn', ', '.join(map(str, CFG_COST['label_cost_usd'])), 'paper_cost_model.yaml', 'giả định'),
    ('L (có huấn luyện)', 'nhãn ban đầu', 'nhãn', '1.550 = 1.200 Train + 350 Validation', 'kích thước tập', 'đo được'),
    ('L (chỉ LLM)', 'nhãn ban đầu', 'nhãn', '367 = 350 Validation + 17 ví dụ few-shot độc nhất (8 tầng 1 + 9 tầng 2; prompt phẳng dùng lại 14 trong số đó)', 'manifest prompt', 'đo được'),
    ('nhãn mỗi lần kiểm tra lại', 'Validation mới (+ Train với mô hình có huấn luyện)', 'nhãn', f"{d['validation_labels_per_check']} (+ {d['extra_train_labels_per_check']})", 'paper_cost_model.yaml', 'giả định'),
    ('k', 'số lần kiểm tra lại', 'lần', str(CM.drift_checks(CFG_COST)), f"mỗi {d['check_every_months']} tháng, chân trời {CFG_COST['horizon_months']} tháng", 'giả định'),
    ('p (GPU)', 'giá GPU', 'USD/giờ', ', '.join(map(str, CFG_COST['gpu_usd_per_hour'])), 'paper_cost_model.yaml', 'giả định'),
    ('p (CPU)', 'giá CPU', 'USD/giờ', str(CFG_COST['cpu_usd_per_hour']), 'paper_cost_model.yaml', 'giả định'),
    ('τ', 'giây huấn luyện một lần chạy', 'giây', 'theo cấu hình', 'paper_cost_components.csv', 'đo được'),
    ('t', 'giây suy luận một review', 'giây', 'theo cấu hình', 'paper_cost_components.csv', 'đo được (khác phần cứng)'),
    ('u', 'USD API một review', 'USD', f"cascade LLM {COMP2.set_index('cấu hình').loc['Cascade LLM few-shot', 'USD API / review'] * 1000:.3f}/1.000; LLM phẳng {COMP2.set_index('cấu hình').loc['Phẳng LLM few-shot', 'USD API / review'] * 1000:.3f}/1.000", 'token API trả về × giá 0,15/0,60 USD mỗi triệu token', 'đo được'),
    ('e', 'máy chủ bật liên tục', '0/1', '0 theo lô, 1 bật liên tục (730 giờ/tháng)', 'paper_cost_model.yaml', 'giả định'),
    ('V', 'khối lượng mỗi tháng', 'review/tháng', ', '.join(map(str, CFG_COST['monthly_volume'])), 'paper_cost_model.yaml', 'giả định'),
    ('chưa tính', 'gán đôi, phân xử, kiểm toán, công chuyên viên, hậu quả bỏ sót', '', 'nằm trong đơn giá nhãn giả định hoặc ngoài phạm vi', '', 'giới hạn'),
], columns=['ký hiệu', 'đại lượng', 'đơn vị', 'giá trị', 'nguồn', 'trạng thái'])
print(PARAMS.to_string(index=False))

def decisions(comp, label_cost=B['label_cost_usd'], gpu=B['gpu_usd_per_hour'], targets=(60, 70, 80), volumes=None):
    cfg = dict(CFG_COST)
    if volumes is not None:
        cfg['monthly_volume'] = volumes
    rows = []
    for m in cfg['deployment']:
        for v in cfg['monthly_volume']:
            sc = CM.tco(comp, cfg, v, label_cost, gpu, m)
            for q in targets:
                for allow in (True, False):
                    s = sc[(sc['Macro-F1'] >= q) & (allow | ~sc['gọi API'])]
                    if s.empty:
                        best = '—'
                    else:
                        mn = s['tổng USD'].min()
                        best = s[s['tổng USD'] <= max(mn * 1.01, mn + 10)].sort_values(['Macro-F1', 'tổng USD'], ascending=[False, True]).iloc[0]['mã']
                    rows.append({'triển khai': m, 'review / tháng': v, 'ngưỡng': q, 'được gọi API': allow, 'cấu hình': best})
    return pd.DataFrame(rows)

# (i) tổ chức đã có nhãn ban đầu
comp_i = COMP2.copy(); comp_i['nhãn ban đầu'] = 0
DEC_BASE, DEC_I = decisions(COMP2), decisions(comp_i)
diff_i = DEC_BASE.merge(DEC_I, on=['triển khai', 'review / tháng', 'ngưỡng', 'được gọi API'], suffixes=(' (gốc)', ' (đã có nhãn)'))
changed = diff_i[diff_i['cấu hình (gốc)'] != diff_i['cấu hình (đã có nhãn)']]
print(f'\n(i) Số ô lựa chọn thay đổi khi đã có nhãn ban đầu (ngưỡng 60/70/80, 0,10 USD/nhãn, GPU 0,35 USD/giờ): {len(changed)} / {len(diff_i)}')
if len(changed):
    print(changed.to_string(index=False))

def be_note(x):
    return 'A luôn rẻ hơn' if x == 0 else ('A không bao giờ rẻ hơn' if np.isinf(x) else '')

BE_PAIRS = (('Phẳng PhoBERT', 'Cascade LLM few-shot'), ('Phẳng PhoBERT', 'Phẳng LLM few-shot'),
            ('PhoBERT ⇢ LLM (θ_max)', 'Cascade LLM few-shot'), ('Phẳng PhoBERT', 'PhoBERT ⇢ LLM (θ_max)'))
rows = []
for mode in CFG_COST['deployment']:
    for a, b in BE_PAIRS:
        g = CM.break_even(COMP2, CFG_COST, a, b, B['label_cost_usd'], B['gpu_usd_per_hour'], mode)
        e = CM.break_even(comp_i, CFG_COST, a, b, B['label_cost_usd'], B['gpu_usd_per_hour'], mode)
        rows.append({'triển khai': mode, 'A (cố định)': a, 'B (biến đổi)': b, 'hòa vốn gốc (review/tháng)': g,
                     'hòa vốn khi đã có nhãn (review/tháng)': e, 'ghi chú': be_note(e) or be_note(g)})
BE_I = pd.DataFrame(rows)
print('Điểm hòa vốn khi đã có nhãn ban đầu:'); print(BE_I.round(0).to_string(index=False))

# (ii) giá LLM nhân hệ số
rows = []
for mult in SEL['cost']['llm_price_multipliers']:
    comp_m = COMP2.copy(); comp_m['USD API / review'] *= mult
    for mode in CFG_COST['deployment']:
        for a, b in BE_PAIRS:
            x = CM.break_even(comp_m, CFG_COST, a, b, B['label_cost_usd'], B['gpu_usd_per_hour'], mode)
            rows.append({'giá LLM ×': mult, 'triển khai': mode, 'A (cố định)': a, 'B (biến đổi)': b,
                         'hòa vốn (review/tháng)': x, 'ghi chú': be_note(x)})
BE_PRICE = pd.DataFrame(rows)
print('\n(ii) Hòa vốn theo giá LLM (0,10 USD/nhãn, GPU 0,35 USD/giờ):'); print(BE_PRICE.round(0).to_string(index=False))

                  ký hiệu                                                     đại lượng       đơn vị                                                                                                       giá trị                                                nguồn               trạng thái
                      c_L                                              chi phí một nhãn     USD/nhãn                                                                                     0.02, 0.05, 0.1, 0.2, 0.5                                paper_cost_model.yaml                 giả định
        L (có huấn luyện)                                                  nhãn ban đầu         nhãn                                                                          1.550 = 1.200 Train + 350 Validation                                       kích thước tập                  đo được
              L (chỉ LLM)                                                  nhãn ban đầu         nhãn 367 = 350 Validation + 17 ví dụ 


(i) Số ô lựa chọn thay đổi khi đã có nhãn ban đầu (ngưỡng 60/70/80, 0,10 USD/nhãn, GPU 0,35 USD/giờ): 0 / 48
Điểm hòa vốn khi đã có nhãn ban đầu:
triển khai           A (cố định)          B (biến đổi)  hòa vốn gốc (review/tháng)  hòa vốn khi đã có nhãn (review/tháng)       ghi chú
     batch         Phẳng PhoBERT  Cascade LLM few-shot                     59889.0                                12084.0              
     batch         Phẳng PhoBERT    Phẳng LLM few-shot                     46435.0                                 9359.0              
     batch PhoBERT ⇢ LLM (θ_max)  Cascade LLM few-shot                     75149.0                                15172.0              
     batch         Phẳng PhoBERT PhoBERT ⇢ LLM (θ_max)                         0.0                                    0.0 A luôn rẻ hơn
  endpoint         Phẳng PhoBERT  Cascade LLM few-shot                   1297624.0                              1249865.0              
  endpoint         Phẳng PhoBERT    P

In [9]:
# (iii) định tuyến với tỷ lệ gọi thực tế từng tập (đã gồm chi phí chạy PhoBERT trong 'giây suy luận / review')
rate_h = float(B2_TAB['holdout'].set_index('cấu hình').loc[CONF_MAX, 'gửi LLM (seed 42)'])
rate_l = float(B2_TAB['lazada'].set_index('cấu hình').loc[CONF_MAX, 'gửi LLM (seed 42)'])
base_row = COMP2.set_index('cấu hình').loc[CONF_MAX]
pho_sec = float(COMP2.set_index('cấu hình').loc['Phẳng PhoBERT', 'giây suy luận / review'])
llm_sec = (float(base_row['giây suy luận / review']) - pho_sec) / max(rate_h, 1e-9)
r_l = base_row.copy(); r_l['USD API / review'] = rate_l * LLM_USD['lazada']; r_l['giây suy luận / review'] = pho_sec + rate_l * llm_sec
comp_iii = pd.concat([COMP2, pd.DataFrame([{**r_l.to_dict(), 'cấu hình': CONF_MAX + ' [tỷ lệ Lazada]', 'mã': 'P⇢L θ_max Laz'}])], ignore_index=True)
for c in ('gọi API', 'có huấn luyện'):
    comp_iii[c] = comp_iii[c].astype(bool)
rows = []
for mode in CFG_COST['deployment']:
    for v in (5_000, 50_000, 500_000):
        sc = CM.tco(comp_iii, CFG_COST, v, B['label_cost_usd'], B['gpu_usd_per_hour'], mode).set_index('cấu hình')
        rows.append({'triển khai': mode, 'review / tháng': v, **{n: sc.loc[n, 'tổng USD'] for n in
                     ('Phẳng PhoBERT', CONF_MAX, CONF_MAX + ' [tỷ lệ Lazada]', 'Cascade LLM few-shot', 'Phẳng LLM few-shot')}})
ROUTE_RATE = pd.DataFrame(rows)
print(f'(iii) Tỷ lệ gọi LLM seed 42: Hold-out {rate_h * 100:.1f}%, Lazada {rate_l * 100:.1f}% | USD/review LLM phẳng: Hold-out {LLM_USD["holdout"] * 1000:.3f}, Lazada {LLM_USD["lazada"] * 1000:.3f} /1.000')
print(ROUTE_RATE.round(0).to_string(index=False))

# (iv) quy mô quan sát
V_OBS = SEL['cost']['observed_reviews_per_year'] / 12
rows = []
for n in ('Phẳng TF-IDF', 'Phẳng PhoBERT', CONF_MAX, 'Cascade LLM few-shot', 'Phẳng LLM few-shot'):
    r = COMP2.set_index('cấu hình').loc[n]
    tco = CM.tco(COMP2[COMP2['cấu hình'] == n], CFG_COST, V_OBS, B['label_cost_usd'], B['gpu_usd_per_hour'], 'batch').iloc[0]
    rows.append({'cấu hình': n, 'Macro-F1 (seed 42)': r['Macro-F1'], 'USD API / tháng': V_OBS * r['USD API / review'],
                 'tổng 12 tháng (USD, theo lô)': tco['tổng USD'], 'trong đó cố định': tco['USD cố định']})
SCALE = pd.DataFrame(rows)
print(f'\n(iv) Quy mô quan sát {V_OBS:,.0f} review/tháng (57.200/năm), 0,10 USD/nhãn, theo lô:'); print(SCALE.round(2).to_string(index=False))

# (v) độ nhạy ngưỡng chất lượng và nguồn Macro-F1
mean_f1 = pd.read_csv(os.path.join(RES, 'paper_step2_e2e_holdout.csv'), encoding='utf-8-sig').set_index('cấu hình')['Macro-F1 (4 lớp)'].astype(str).str.split(' ±').str[0].astype(float).to_dict()
r4 = pd.read_csv(os.path.join(RES, 'paper_step4_routing_holdout.csv'), encoding='utf-8-sig').set_index('cấu hình')['Macro-F1 (trung bình)'].mul(100).to_dict()
comp_mean = COMP2.copy(); comp_mean['Macro-F1'] = comp_mean['cấu hình'].map({**mean_f1, **r4}).fillna(comp_mean['Macro-F1'])
tq = SEL['cost']['quality_targets']
D42, DMEAN = decisions(COMP2, targets=tq), decisions(comp_mean, targets=tq)
SENS = D42.merge(DMEAN, on=['triển khai', 'review / tháng', 'ngưỡng', 'được gọi API'], suffixes=(' (seed 42)', ' (TB 3 seed)'))
SENS = SENS[SENS['được gọi API']]
print('\n(v) Cấu hình rẻ nhất theo ngưỡng 78–80, Macro-F1 seed 42 so với trung bình ba seed (được gọi API):')
print(SENS.pivot_table(index=['triển khai', 'review / tháng'], columns='ngưỡng', values=['cấu hình (seed 42)', 'cấu hình (TB 3 seed)'], aggfunc='first').to_string())

(iii) Tỷ lệ gọi LLM seed 42: Hold-out 12.2%, Lazada 19.2% | USD/review LLM phẳng: Hold-out 0.266, Lazada 0.268 /1.000
triển khai  review / tháng  Phẳng PhoBERT  PhoBERT ⇢ LLM (θ_max)  PhoBERT ⇢ LLM (θ_max) [tỷ lệ Lazada]  Cascade LLM few-shot  Phẳng LLM few-shot
     batch            5000          220.0                  223.0                                 224.0                  84.0                88.0
     batch           50000          220.0                  245.0                                 260.0                 196.0               232.0
     batch          500000          221.0                  472.0                                 618.0                1310.0              1668.0
  endpoint            5000         3286.0                 3288.0                                3289.0                  84.0                88.0
  endpoint           50000         3286.0                 3306.0                                3317.0                 196.0               232.0
  endpoint  


(v) Cấu hình rẻ nhất theo ngưỡng 78–80, Macro-F1 seed 42 so với trung bình ba seed (được gọi API):
                          cấu hình (TB 3 seed)                       cấu hình (seed 42)                
ngưỡng                                      78         79         80                 78         79   80
triển khai review / tháng                                                                              
batch      1000                            L→L        L→L        L→L                L→L        L→L  L→L
           10000                           L→L        L→L        L→L                L→L        L→L  L→L
           100000                    P⇢L θ_max  P⇢L θ_max  P⇢L θ_max          P⇢L θ_max  P⇢L θ_max  L→L
           1000000                    P⇢L θ_98   P⇢L θ_98  P⇢L θ_max           P⇢L θ_98   P⇢L θ_98  L→L
endpoint   1000                            L→L        L→L        L→L                L→L        L→L  L→L
           10000                           L→L        L→L        L→L

In [10]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'paper_step5_{name}.csv'), index=False, encoding='utf-8-sig')

save(B1_VAL, 'b1_validation')
save(pd.concat(B1_TAB.values(), ignore_index=True), 'b1_results')
save(pd.concat([v for v in B1_CMP.values() if v is not None], ignore_index=True), 'b1_comparisons')
save(pd.concat(B2_TAB.values(), ignore_index=True), 'b2_results')
save(pd.concat(B2_RND.values(), ignore_index=True), 'b2_random_baseline')
save(pd.concat(B2_CMP.values(), ignore_index=True), 'b2_comparisons')
save(NEAR_SUM, 'near_duplicates_summary'); save(NEAR_E2E, 'near_duplicates_holdout_e2e')
save(PARAMS, 'cost_parameters'); save(diff_i, 'cost_existing_labels'); save(BE_I, 'cost_existing_labels_break_even')
save(BE_PRICE, 'cost_llm_price_break_even')
save(ROUTE_RATE, 'cost_routing_rates'); save(SCALE, 'cost_observed_scale'); save(SENS, 'cost_target_sensitivity')
json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'python': sys.version.split()[0], 'amendment': 'v4.2', 'selection_sha256': SEL_HASH,
           'n_boot': 10000, 'bootstrap_seed': 42, 'random_draws': FC.RANDOM_DRAWS, 'random_seed': FC.RANDOM_SEED, 'holdout_use': 4},
          open(os.path.join(RES, 'paper_step5_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Đã lưu', sorted(f for f in os.listdir(RES) if f.startswith('paper_step5_')))

Đã lưu ['paper_step5_b1_comparisons.csv', 'paper_step5_b1_results.csv', 'paper_step5_b1_validation.csv', 'paper_step5_b2_comparisons.csv', 'paper_step5_b2_random_baseline.csv', 'paper_step5_b2_results.csv', 'paper_step5_cost_existing_labels.csv', 'paper_step5_cost_existing_labels_break_even.csv', 'paper_step5_cost_llm_price_break_even.csv', 'paper_step5_cost_observed_scale.csv', 'paper_step5_cost_parameters.csv', 'paper_step5_cost_routing_rates.csv', 'paper_step5_cost_target_sensitivity.csv', 'paper_step5_near_duplicates_holdout_e2e.csv', 'paper_step5_near_duplicates_summary.csv', 'paper_step5_run_info.json', 'paper_step5_selection.json']
